In [ ]:
import jax
import jax.numpy as jnp
import flax.nnx as nnx
import netket as nk
import netket.experimental as nkx

import optax
from typing import Callable
from functools import partial
from jax.flatten_util import ravel_pytree
from pyscf import gto, scf, fci
import time
from NES_VMC_H2_631G import get_ccsd_excitations_and_sampler_edges_from_hf
from NES_VMC import NESTotalAnsatz,NESTotalAnsatz_stable,create_machine,\
        SingleStateAnsatz,create_single_machine,create_machine_stable,\
            create_machine_matrix_stable,create_machine_max_stable,\
        create_machine_matrix,NES_loss_energy,nes_vmc_gradient,nes_vmc_gradient_stable,\
        compute_qgt,sampler_info,NESFermionHopRule,\
        create_machine_stable,create_machine_matrix_stable,create_gauge_fixed_total_machines
        
import logging
# ========== 你原有全局参数（直接复用） ==========
bond_length = 1.8
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='6-31G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)
hf_ground_energy = mf.e_tot
print(f'HF 基准能量: {hf_ground_energy:.8f}')

cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()
print("="*60)
print("H₂ FCI 基准能量")
print("="*60)
for i, e in enumerate(E_fcis):
    exc = (e - E_fcis[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能: {exc:.4f} eV")

# ha = nkx.operator.from_pyscf_molecule(mol)
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=4,
    s=1/2,
    n_fermions_per_spin=(1,1),
)
K = 2  # NES 扩展副本数
hi_ext = hi ** K  # 扩展希尔伯特空间
ha = nkx.operator.from_pyscf_molecule(mol)
Hatree_Fock = hi.all_states()[0]
single_edges, singles, doubles = get_ccsd_excitations_and_sampler_edges_from_hf(
    Hatree_Fock
)
print(f'single_edges: {single_edges}')
g = nk.graph.Graph(edges=single_edges)
single_rule = nk.sampler.rules.FermionHopRule(hilbert=hi, graph=g)
tensor_rule = nk.sampler.rules.TensorRule(hi_ext, [single_rule] * K)
#sampler = nk.sampler.MetropolisSampler(hi, rule=single_rule, n_chains=100, sweep_size=32)

SINGLE_SIZE = hi.size
ext_edges = []
for k in range(K):
    offset = k * SINGLE_SIZE
    for (i, j) in single_edges:
        ext_edges.append((i + offset, j + offset))
ext_edges = jnp.array(ext_edges)  # 转为jax数组（关键修复）
ext_edges

nes_rule = NESFermionHopRule(edges=ext_edges, K=K, single_size=SINGLE_SIZE)

jnp.set_printoptions(
    linewidth=9999,   # 单行宽度拉满，绝不自动换行
    threshold=jnp.inf, # 全部打印，不省略
    precision=8,      # 小数位数按需调整
    suppress=False
)

/opt/miniconda3/envs/Netket/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


∣NK⟩ Tip: Prefer the new nk.driver.VMC_SR over VMC which supports minSR and SPRING.

E0 = -1.02613572 Ha  |  激发能: 0.0000 eV
E1 = -0.97892204 Ha  |  激发能: 1.2848 eV
E2 = -0.66776157 Ha  |  激发能: 9.7519 eV
E3 = -0.60817046 Ha  |  激发能: 11.3734 eV


In [2]:
import logging
# 日志配置
logger = logging.getLogger('NES_VMC_K4')
logger.setLevel(logging.INFO)
# 阻止日志向上传播
logger.propagate = False
# 清除所有旧handler，防止重复打印
logger.handlers.clear()

# 自定义日志格式：只打印内容，不带等级、logger名
simple_formatter = logging.Formatter("%(message)s", datefmt="%H:%M:%S")

# 1. 文件输出处理器
file_handler = logging.FileHandler("nes_vmc_0626_K2.log", mode="w", encoding="utf-8")
file_handler.setFormatter(simple_formatter)
file_handler.setLevel(logging.INFO)
logger.addHandler(file_handler)

console_handler = logging.StreamHandler()
console_handler.setFormatter(simple_formatter)
console_handler.setLevel(logging.INFO)
logger.addHandler(console_handler)

print('库导入完成')

库导入完成


In [4]:
import time
import jax
import jax.numpy as jnp
import optax

# ====================== 超参统一配置 ======================
N_CHAINS = 16*K
N_WARMUP = 100
N_SAMPLES_PER_CHAIN = 200
SWEEP_SIZE = 30
N_ITER = 200
SINGLE_SIZE = hi.size  # 单个子系统维度 = 4
Natural_Grad = False
clip_norm = 2.0        # 全局梯度L2上限，QML推荐1~2
lr = 0.01
qgt_diag_shift = 0.1  # 上调正则，抑制QGT梯度爆炸

# ====================== 模型初始化 ======================
total_ansatz = NESTotalAnsatz_stable(SINGLE_SIZE, K, 12, rngs=nnx.Rngs(11))
total_machine, total_graphdef, total_params = create_machine_stable(total_ansatz)
total_matrix_machine, _, _ = create_machine_matrix_stable(total_ansatz)
total_max_machine, _, _ = create_machine_max_stable(total_ansatz)


single_machine_list = []
for ansatz in total_ansatz.single_ansatz_list:
    m, _, _ = create_single_machine(ansatz)
    single_machine_list.append(m)
    

# ====================== 优化器：梯度裁剪 + SGD ======================
# chain顺序：先裁剪梯度，再SGD更新
optimizer = optax.chain(
    optax.clip_by_global_norm(clip_norm),
    optax.sgd(learning_rate=lr)
)
opt_state = optimizer.init(total_params)

# ====================== 扩展采样边、自定义采样器 ======================
ext_edges = []
for k in range(K):
    offset = k * SINGLE_SIZE
    for (i, j) in single_edges:
        ext_edges.append((i + offset, j + offset))
ext_edges = jnp.array(ext_edges)

nes_rule = NESFermionHopRule(edges=ext_edges, K=K, single_size=SINGLE_SIZE)
nes_sampler = nk.sampler.MetropolisSampler(
    hilbert=hi_ext,
    rule=nes_rule,
    n_chains=N_CHAINS,
    sweep_size=SWEEP_SIZE
)

sampler_rng = jax.random.PRNGKey(21)
sampler_state = nes_sampler.init_state(total_machine, total_params, sampler_rng)

# ====================== 训练历史（新增Ψ矩阵条件数监控） ======================
history = {
    'step': [],
    'energy_0st': [],
    'energy_1st': [],
    'energy_2st': [],
    'energy_3st': [],
    'loss': [],
    'params': [],
    'E_Lmatrix': [],
    'samples': [],
    'log_Psi_mean': [],
    'log_Psi_min': [],
    'log_Psi_max': [],
    'grad_norm_raw': [],       # QGT前原始梯度
    'grad_norm_natural': [],   # QGT自然梯度（裁剪前）
    'grad_norm_clipped': [],   # 裁剪后真实梯度（≤clip_norm）
    'psi_cond': [],           # 新增：波函数矩阵条件数
}

logger.info("\n" + "="*60)
logger.info(f"开始多链 NES-VMC 训练 | 使用{'自然' if Natural_Grad else '原始'}梯度")
logger.info("="*60)
logger.info(f"FCI基准：基态={E_fcis[0]:.8f} Ha | 1激发={E_fcis[1]:.8f} Ha ")
logger.info(f"理论 Loss 上限：{sum(E_fcis[0:2]):.8f} ")
logger.info(f"超参：clip_norm={clip_norm}, lr={lr}, QGT diag_shift={qgt_diag_shift}")

start_time = time.time()
for step in range(N_ITER):
    # 采样
    samples_raw, sampler_state = nes_sampler.sample(
        machine=total_machine,
        parameters=total_params,
        state=sampler_state,
        chain_length=N_SAMPLES_PER_CHAIN
    )
    samples = samples_raw.reshape(-1, hi_ext.size)
    x_batch = samples.reshape(-1, K, SINGLE_SIZE)

    # 1. 原始变分梯度
    grad_raw, loss_mean, E_L_mean = nes_vmc_gradient_stable(
        ha=ha,
        total_matrix_machine=total_matrix_machine,
        total_max_machine=total_max_machine,
        total_machine=total_machine,
        single_machine_list=single_machine_list,
        total_params=total_params,
        x_batch=x_batch
    )
    grad_raw_flat, unravel_fn = ravel_pytree(grad_raw)
    grad_norm_raw = jnp.linalg.norm(grad_raw_flat)
    grad_update = grad_raw

    # ========== 异常提前拦截，防止崩溃卡死 ==========
    has_nan_grad = jnp.any(jnp.isnan(grad_raw_flat))
    grad_explode = grad_norm_raw > 5000.0
    if has_nan_grad or grad_explode:
        logger.warning(f"【Step {step} 告警】梯度异常！nan={has_nan_grad}, raw_grad_norm={grad_norm_raw:.2f}")

    # 2. QGT自然梯度预条件
    if Natural_Grad:
        qgt_reg_mat, _ = compute_qgt(
            total_machine, total_params, x_batch, diag_shift=qgt_diag_shift
        )
        ng_flat = jnp.linalg.solve(qgt_reg_mat, grad_raw_flat)
        grad_update = unravel_fn(ng_flat)
        grad_norm_natural = jnp.linalg.norm(ng_flat)
    else:
        grad_norm_natural = grad_norm_raw

    # 3. 梯度裁剪（optimizer.update内部自动执行）
    updates, opt_state = optimizer.update(grad_update, opt_state, total_params)
    # 单独计算裁剪后梯度范数用于监控
    clip_transform = optax.clip_by_global_norm(clip_norm)
    clipped_grad, _ = clip_transform.update(grad_update, opt_state[0], total_params)
    clipped_flat, _ = ravel_pytree(clipped_grad)
    grad_norm_clipped = jnp.linalg.norm(clipped_flat)
    
    # 如果梯度范数为0 则结束迭代
    if grad_norm_clipped == 0.0:
        logger.info(f"【Step {step} 告警】梯度范数为0，结束迭代")
        break

    # 参数更新
    total_params = optax.apply_updates(total_params, updates)

    # ====================== 新增：计算Ψ矩阵条件数 ======================
    # 取单批次样本计算波函数矩阵，用第一组组态做代表
    x_single = x_batch[0:1, ...]
    psi_mat = total_matrix_machine(total_params, x_single)[0]  # 取出N×N波函数矩阵
    psi_cond = jnp.linalg.cond(psi_mat)

    # 波函数标量输出
    log_Psi_batch = total_machine(total_params, x_batch)
    eig_vals, eig_vecs = jnp.linalg.eig(E_L_mean)
    
    
    # eig_vals, eig_vecs = jnp.linalg.eig(E_L_mean)
    # sort_idx = jnp.argsort(eig_vals.real)
    # eig_vals, eig_vecs = eig_vals[sort_idx], eig_vecs[:, sort_idx]

    # 记录历史
    history['step'].append(step)
    history['loss'].append(loss_mean)
    history['E_Lmatrix'].append(E_L_mean)
    history['samples'].append(samples)
    history['log_Psi_mean'].append(log_Psi_batch.mean())
    history['log_Psi_min'].append(log_Psi_batch.min())
    history['log_Psi_max'].append(log_Psi_batch.max())
    history['energy_0st'].append(eig_vals[0])
    history['energy_1st'].append(eig_vals[1])
    history['energy_2st'].append(eig_vals[2])
    history['energy_3st'].append(eig_vals[3])
    history['params'].append(total_params)
    history['grad_norm_raw'].append(grad_norm_raw)
    history['grad_norm_natural'].append(grad_norm_natural)
    history['grad_norm_clipped'].append(grad_norm_clipped)
    history['psi_cond'].append(psi_cond)  # 保存条件数

    # 打印日志，新增Ψ条件数输出
    if step % 10 == 0 or step == N_ITER - 1:
        logger.info(f"[Step {step:3d}] logΨ: mean={log_Psi_batch.mean():.3f} | min={log_Psi_batch.min():.3f} | max={log_Psi_batch.max():.3f}")
        logger.info(f"梯度监控 | raw={grad_norm_raw:.4f} | natural={grad_norm_natural:.4f} | clipped={grad_norm_clipped:.4f}(上限{clip_norm})")
        logger.info(f"Ψ矩阵条件数 cond(Ψ) = {psi_cond:.2e}")
        logger.info(f"Loss={loss_mean:.6f} | E0={eig_vals[0]:.8f} | E1={eig_vals[1]:.8f}")
        logger.info("#-----------------------------------------#")

end_time = time.time()
print(f"训练耗时：{end_time - start_time:.2f} 秒")
print("\n" + "="*60)
print("训练完成!")
print("="*60)


开始多链 NES-VMC 训练 | 使用原始梯度
FCI基准：基态=-1.02613572 Ha | 1激发=-0.97892204 Ha 
理论 Loss 上限：-2.00505776 
超参：clip_norm=2.0, lr=0.01, QGT diag_shift=0.1
[Step   0] logΨ: mean=0.016-0.231j | min=-3.565+2.168j | max=0.853+0.457j
梯度监控 | raw=1.9810 | natural=1.9810 | clipped=1.9810(上限2.0)
Ψ矩阵条件数 cond(Ψ) = 1.40e+01
Loss=0.580044 | E0=0.60571557+0.00295268j | E1=-0.02567199-0.00493506j
#-----------------------------------------#
[Step  10] logΨ: mean=0.618-0.508j | min=-2.344+0.526j | max=1.466+0.477j
梯度监控 | raw=1.6564 | natural=1.6564 | clipped=1.6564(上限2.0)
Ψ矩阵条件数 cond(Ψ) = 2.26e+00
Loss=-0.158555 | E0=0.42889011-0.00205933j | E1=-0.58744526-0.00039874j
#-----------------------------------------#
[Step  20] logΨ: mean=5.907-1.206j | min=-1.468+0.449j | max=6.894+0.494j
梯度监控 | raw=4.2204 | natural=4.2204 | clipped=2.0000(上限2.0)
Ψ矩阵条件数 cond(Ψ) = 1.99e+00
Loss=-1.070564 | E0=-0.90945554+0.00155179j | E1=-0.16110824-0.00004660j
#-----------------------------------------#
[Step  30] logΨ: mean=4.902+0.820

训练耗时：188.71 秒

训练完成!


In [6]:
# ============================================================
# NES-VMC Stable Training Cell | K=2 | H2 / 6-31G
# 带完整诊断：梯度消失 / amplitude collapse / gauge plateau
# ============================================================

import os
import time
import logging
import numpy as np

import jax
import jax.numpy as jnp
import flax.nnx as nnx
import netket as nk
import netket.experimental as nkx
import optax

from jax.flatten_util import ravel_pytree
from pyscf import gto, scf, fci

# 如果你已经在 notebook 前面 import 过，也可以不重复 import
from NES_VMC_H2_631G import get_ccsd_excitations_and_sampler_edges_from_hf

from NES_VMC import (
    NESTotalAnsatz_stable,
    create_machine_stable,
    create_machine_matrix_stable,
    create_machine_max_stable,
    create_single_machine,
    nes_vmc_gradient_stable,
    NES_loss_energy_stable,      # 如果你模块里名字不同，就改成你实际稳定版 loss 函数名
    compute_qgt,
    NESFermionHopRule,
)

# 小系统建议打开 x64，别为了省一点算力把数值稳定性献祭了
jax.config.update("jax_enable_x64", True)

jnp.set_printoptions(
    linewidth=9999,
    threshold=jnp.inf,
    precision=8,
    suppress=False,
)


# ============================================================
# 0. 工具函数
# ============================================================

def tree_l2_norm(tree):
    leaves = jax.tree_util.tree_leaves(tree)
    if len(leaves) == 0:
        return jnp.array(0.0)
    return jnp.sqrt(
        sum([jnp.sum(jnp.abs(x) ** 2) for x in leaves])
    )


def tree_all_finite(tree):
    leaves = jax.tree_util.tree_leaves(tree)
    if len(leaves) == 0:
        return True
    flags = [jnp.all(jnp.isfinite(x)) for x in leaves]
    return bool(jnp.all(jnp.asarray(flags)))


def unique_ratio_from_samples(samples):
    """
    samples: shape (n_samples, hi_ext.size)
    """
    arr = np.asarray(samples)
    arr = arr.reshape(arr.shape[0], -1)
    unique = len({tuple(row.tolist()) for row in arr})
    return unique / max(arr.shape[0], 1)


def tree_batch_std_norm(tree, batch_size):
    """
    计算 dlogΨ(params, x) 在 batch 维度上的变化强度。
    如果这个接近 0，说明不同样本上的参数响应几乎一样，
    covariance 梯度会自然消失。
    """
    leaves = jax.tree_util.tree_leaves(tree)
    total = 0.0

    for leaf in leaves:
        if leaf.ndim >= 1 and leaf.shape[0] == batch_size:
            centered = leaf - jnp.mean(leaf, axis=0, keepdims=True)
            total = total + jnp.sum(jnp.abs(centered) ** 2)

    return jnp.sqrt(total)


def tree_batch_mean_norm(tree, batch_size):
    leaves = jax.tree_util.tree_leaves(tree)
    total = 0.0

    for leaf in leaves:
        if leaf.ndim >= 1 and leaf.shape[0] == batch_size:
            mean_leaf = jnp.mean(leaf, axis=0)
            total = total + jnp.sum(jnp.abs(mean_leaf) ** 2)

    return jnp.sqrt(total)


def safe_real(x):
    return float(jnp.real(x))


def safe_float(x):
    return float(jnp.asarray(x))


# ============================================================
# 1. 分子、Hilbert、FCI 基准
# ============================================================

bond_length = 1.8
geometry = [
    ("H", (0.0, 0.0, 0.0)),
    ("H", (bond_length, 0.0, 0.0)),
]

mol = gto.M(atom=geometry, basis="6-31G", verbose=0)
mf = scf.RHF(mol).run(verbose=0)
hf_ground_energy = mf.e_tot

cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()

print("=" * 60)
print("H2 / 6-31G 基准")
print("=" * 60)
print(f"HF energy = {hf_ground_energy:.8f} Ha")
for i, e in enumerate(E_fcis):
    exc = (e - E_fcis[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha | excitation = {exc:.4f} eV")

hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=4,
    s=1/2,
    n_fermions_per_spin=(1, 1),
)

K = 2
hi_ext = hi ** K
ha = nkx.operator.from_pyscf_molecule(mol)

SINGLE_SIZE = hi.size

print("=" * 60)
print("Hilbert 信息")
print("=" * 60)
print(f"K = {K}")
print(f"hi.size = {hi.size}")
print(f"hi_ext.size = {hi_ext.size}")
print(f"SINGLE_SIZE = {SINGLE_SIZE}")

target_loss = float(np.sum(E_fcis[:K]))
print(f"target_loss = sum(E_fcis[:K]) = {target_loss:.8f}")

assert K == 2
assert hi_ext.size == K * SINGLE_SIZE


# ============================================================
# 2. Sampler edges
# ============================================================

Hatree_Fock = hi.all_states()[0]

single_edges, singles, doubles = get_ccsd_excitations_and_sampler_edges_from_hf(
    Hatree_Fock
)

print("=" * 60)
print("Sampler edges")
print("=" * 60)
print(f"single_edges = {single_edges}")
print(f"n_singles = {len(singles)}")
print(f"n_doubles = {len(doubles)}")

ext_edges = []
for k in range(K):
    offset = k * SINGLE_SIZE
    for i, j in single_edges:
        ext_edges.append((i + offset, j + offset))

ext_edges = jnp.asarray(ext_edges)

nes_rule = NESFermionHopRule(
    edges=ext_edges,
    K=K,
    single_size=SINGLE_SIZE,
)


# ============================================================
# 3. 日志配置
# ============================================================

logger = logging.getLogger(f"NES_VMC_K{K}_diagnostic")
logger.setLevel(logging.INFO)
logger.propagate = False
logger.handlers.clear()

simple_formatter = logging.Formatter("%(message)s")

log_filename = f"nes_vmc_H2_631G_K{K}_diagnostic.log"

file_handler = logging.FileHandler(log_filename, mode="w", encoding="utf-8")
file_handler.setFormatter(simple_formatter)
file_handler.setLevel(logging.INFO)
logger.addHandler(file_handler)

console_handler = logging.StreamHandler()
console_handler.setFormatter(simple_formatter)
console_handler.setLevel(logging.INFO)
logger.addHandler(console_handler)

logger.info("\n" + "=" * 80)
logger.info("NES-VMC Stable Training | Diagnostic Version")
logger.info("=" * 80)
logger.info(f"K = {K}")
logger.info(f"SINGLE_SIZE = {SINGLE_SIZE}")
logger.info(f"target_loss = {target_loss:.8f}")
logger.info(f"FCI E0 = {E_fcis[0]:.8f} | E1 = {E_fcis[1]:.8f}")


# ============================================================
# 4. 超参数
# ============================================================

N_CHAINS = 16 * K
N_WARMUP = 50
N_SAMPLES_PER_CHAIN = 200
SWEEP_SIZE = 30
N_ITER = 300

Natural_Grad = False

lr = 0.01
clip_norm = 1.0

# 原始梯度超过这个阈值，直接 skip，不是 warning 后继续送死
grad_skip_threshold = 10.0

qgt_diag_shift = 0.1

# 每隔多少步完整诊断一次
PRINT_EVERY = 10

# dlogΨ 诊断比较贵，只取一小部分样本
DIAG_BATCH_SIZE = 128

logger.info("=" * 80)
logger.info("Hyperparameters")
logger.info("=" * 80)
logger.info(f"N_CHAINS = {N_CHAINS}")
logger.info(f"N_WARMUP = {N_WARMUP}")
logger.info(f"N_SAMPLES_PER_CHAIN = {N_SAMPLES_PER_CHAIN}")
logger.info(f"SWEEP_SIZE = {SWEEP_SIZE}")
logger.info(f"N_ITER = {N_ITER}")
logger.info(f"Natural_Grad = {Natural_Grad}")
logger.info(f"lr = {lr}")
logger.info(f"clip_norm = {clip_norm}")
logger.info(f"grad_skip_threshold = {grad_skip_threshold}")
logger.info(f"qgt_diag_shift = {qgt_diag_shift}")
logger.info(f"DIAG_BATCH_SIZE = {DIAG_BATCH_SIZE}")


# ============================================================
# 5. 模型初始化
# ============================================================

hidden_dim = 12
rng_seed = 11

total_ansatz = NESTotalAnsatz_stable(
    SINGLE_SIZE,
    K,
    hidden_dim,
    rngs=nnx.Rngs(rng_seed),
)

total_machine, total_graphdef, total_params = create_machine_stable(total_ansatz)
total_matrix_machine, _, _ = create_machine_matrix_stable(total_ansatz)
total_max_machine, _, _ = create_machine_max_stable(total_ansatz)

single_machine_list = []
for ansatz in total_ansatz.single_ansatz_list:
    m, _, _ = create_single_machine(ansatz)
    single_machine_list.append(m)

assert len(single_machine_list) == K


# ============================================================
# 6. Optimizer
# ============================================================

optimizer = optax.chain(
    optax.clip_by_global_norm(clip_norm),
    optax.sgd(learning_rate=lr),
)

opt_state = optimizer.init(total_params)


# ============================================================
# 7. Sampler
# ============================================================

nes_sampler = nk.sampler.MetropolisSampler(
    hilbert=hi_ext,
    rule=nes_rule,
    n_chains=N_CHAINS,
    sweep_size=SWEEP_SIZE,
)

sampler_rng = jax.random.PRNGKey(21)
sampler_state = nes_sampler.init_state(
    total_machine,
    total_params,
    sampler_rng,
)

# warmup
logger.info("=" * 80)
logger.info("Warmup")
logger.info("=" * 80)

for _ in range(N_WARMUP):
    _, sampler_state = nes_sampler.sample(
        machine=total_machine,
        parameters=total_params,
        state=sampler_state,
        chain_length=1,
    )

logger.info("Warmup done.")


# ============================================================
# 8. 诊断函数
# ============================================================

def compute_diagnostics(
    total_params,
    x_batch,
    samples,
    E_L_mean,
):
    """
    返回当前参数下的完整诊断量。
    注意：这里会额外调用 NES_loss_energy_stable 来得到 E_L_batch。
    """

    n_total = x_batch.shape[0]
    n_diag = min(DIAG_BATCH_SIZE, n_total)

    x_diag = x_batch[:n_diag]
    samples_diag = samples[:n_diag]

    # ---------- logΨ 统计 ----------
    log_Psi_batch = jax.vmap(
        lambda xx: total_machine(total_params, xx)
    )(x_batch)

    log_real = jnp.real(log_Psi_batch)
    log_imag = jnp.imag(log_Psi_batch)

    log_real_mean = jnp.mean(log_real)
    log_real_std = jnp.std(log_real)
    log_real_span = jnp.max(log_real) - jnp.min(log_real)

    log_imag_mean = jnp.mean(log_imag)
    log_imag_std = jnp.std(log_imag)
    log_imag_span = jnp.max(log_imag) - jnp.min(log_imag)

    # ---------- L_stable / Ψ_stable 条件数 ----------
    L_stable_diag = total_matrix_machine(total_params, x_diag)
    Psi_stable_diag = jnp.exp(L_stable_diag)

    conds = jax.vmap(jnp.linalg.cond)(Psi_stable_diag)
    psi_cond_first = conds[0]
    psi_cond_mean = jnp.mean(conds)
    psi_cond_max = jnp.max(conds)

    # ---------- shift 统计 ----------
    shifts = total_max_machine(total_params, x_diag)
    shift_mean = jnp.mean(shifts)
    shift_std = jnp.std(shifts)
    shift_span = jnp.max(shifts) - jnp.min(shifts)

    # ---------- E_L_batch / trace 统计 ----------
    loss_batch, E_L_batch, loss_aux = NES_loss_energy_stable(
        ha=ha,
        total_matrix_machine=total_matrix_machine,
        total_max_machine=total_max_machine,
        single_machine_list=single_machine_list,
        total_params=total_params,
        x=x_diag,
        return_aux=True,
    )

    assert E_L_batch.shape[-2:] == (K, K), (
        f"E_L_batch.shape={E_L_batch.shape}, expected (..., {K}, {K})"
    )

    trace_batch = jnp.trace(E_L_batch, axis1=-2, axis2=-1)
    trace_real = jnp.real(trace_batch)
    trace_imag = jnp.imag(trace_batch)

    trace_real_mean = jnp.mean(trace_real)
    trace_real_std = jnp.std(trace_real)
    trace_real_span = jnp.max(trace_real) - jnp.min(trace_real)

    trace_imag_mean = jnp.mean(trace_imag)
    trace_imag_std = jnp.std(trace_imag)
    trace_imag_span = jnp.max(trace_imag) - jnp.min(trace_imag)

    # ---------- valid ratio ----------
    if isinstance(loss_aux, dict) and "valid" in loss_aux:
        valid = loss_aux["valid"]
        valid_ratio = jnp.mean(valid.astype(jnp.float64))
    else:
        valid_ratio = jnp.array(1.0)

    # ---------- E_L_mean Hermiticity ----------
    herm_error = (
        jnp.linalg.norm(E_L_mean - E_L_mean.conj().T)
        / (jnp.linalg.norm(E_L_mean) + 1e-12)
    )

    imag_norm = jnp.linalg.norm(jnp.imag(E_L_mean))

    E_L_herm = 0.5 * (E_L_mean + E_L_mean.conj().T)
    eig_vals_herm = jnp.linalg.eigvalsh(E_L_herm)

    eig_vals_raw = jnp.linalg.eigvals(E_L_mean)
    eig_vals_raw = eig_vals_raw[jnp.argsort(jnp.real(eig_vals_raw))]

    # ---------- sampler unique ratio ----------
    unique_ratio = unique_ratio_from_samples(samples_diag)

    # ---------- dlogΨ batch 方差 ----------
    grad_logPsi = jax.grad(total_machine, argnums=0, holomorphic=True)
    dlogPsi_batch = jax.vmap(
        grad_logPsi,
        in_axes=(None, 0),
    )(total_params, x_diag)

    dlog_std_norm = tree_batch_std_norm(dlogPsi_batch, n_diag)
    dlog_mean_norm = tree_batch_mean_norm(dlogPsi_batch, n_diag)
    dlog_std_ratio = dlog_std_norm / (dlog_mean_norm + 1e-12)

    return {
        "log_Psi_batch": log_Psi_batch,

        "log_real_mean": log_real_mean,
        "log_real_std": log_real_std,
        "log_real_span": log_real_span,

        "log_imag_mean": log_imag_mean,
        "log_imag_std": log_imag_std,
        "log_imag_span": log_imag_span,

        "psi_cond_first": psi_cond_first,
        "psi_cond_mean": psi_cond_mean,
        "psi_cond_max": psi_cond_max,

        "shift_mean": shift_mean,
        "shift_std": shift_std,
        "shift_span": shift_span,

        "trace_real_mean": trace_real_mean,
        "trace_real_std": trace_real_std,
        "trace_real_span": trace_real_span,

        "trace_imag_mean": trace_imag_mean,
        "trace_imag_std": trace_imag_std,
        "trace_imag_span": trace_imag_span,

        "valid_ratio": valid_ratio,

        "herm_error": herm_error,
        "imag_norm": imag_norm,

        "eig_vals_herm": eig_vals_herm,
        "eig_vals_raw": eig_vals_raw,

        "unique_ratio": unique_ratio,

        "dlog_std_norm": dlog_std_norm,
        "dlog_mean_norm": dlog_mean_norm,
        "dlog_std_ratio": dlog_std_ratio,
    }


def log_diagnostics(step, loss_mean, grad_norm_raw, grad_norm_update, grad_norm_clipped, diag):
    eig_vals_herm = diag["eig_vals_herm"]
    eig_vals_raw = diag["eig_vals_raw"]

    energy_herm_str = " | ".join(
        [f"E{i}_herm={eig_vals_herm[i]:.8f}" for i in range(K)]
    )

    energy_raw_str = " | ".join(
        [f"E{i}_raw={eig_vals_raw[i]:.8f}" for i in range(K)]
    )

    logger.info(f"[Step {step:4d}]")
    logger.info(
        f"Loss={loss_mean:.8f} | "
        f"target={target_loss:.8f} | "
        f"gap={float(loss_mean - target_loss):+.8f}"
    )

    logger.info(
        f"Grad | raw={grad_norm_raw:.4e} | "
        f"update={grad_norm_update:.4e} | "
        f"clipped={grad_norm_clipped:.4e} | "
        f"clip_norm={clip_norm:.2e}"
    )

    logger.info(
        "logΨ.real | "
        f"mean={diag['log_real_mean']:.6f} | "
        f"std={diag['log_real_std']:.4e} | "
        f"span={diag['log_real_span']:.4e}"
    )

    logger.info(
        "logΨ.imag | "
        f"mean={diag['log_imag_mean']:.6f} | "
        f"std={diag['log_imag_std']:.4e} | "
        f"span={diag['log_imag_span']:.4e}"
    )

    logger.info(
        "shift | "
        f"mean={diag['shift_mean']:.6f} | "
        f"std={diag['shift_std']:.4e} | "
        f"span={diag['shift_span']:.4e}"
    )

    logger.info(
        "cond(Ψ_stable) | "
        f"first={diag['psi_cond_first']:.4e} | "
        f"mean={diag['psi_cond_mean']:.4e} | "
        f"max={diag['psi_cond_max']:.4e}"
    )

    logger.info(
        "trace(E_L).real | "
        f"mean={diag['trace_real_mean']:.8f} | "
        f"std={diag['trace_real_std']:.4e} | "
        f"span={diag['trace_real_span']:.4e}"
    )

    logger.info(
        "trace(E_L).imag | "
        f"mean={diag['trace_imag_mean']:.8f} | "
        f"std={diag['trace_imag_std']:.4e} | "
        f"span={diag['trace_imag_span']:.4e}"
    )

    logger.info(
        "E_L_mean diagnostics | "
        f"herm_error={diag['herm_error']:.4e} | "
        f"imag_norm={diag['imag_norm']:.4e} | "
        f"valid_ratio={diag['valid_ratio']:.4f}"
    )

    logger.info(
        "sampler / dlogΨ | "
        f"unique_ratio={diag['unique_ratio']:.4f} | "
        f"dlog_std_norm={diag['dlog_std_norm']:.4e} | "
        f"dlog_mean_norm={diag['dlog_mean_norm']:.4e} | "
        f"dlog_std_ratio={diag['dlog_std_ratio']:.4e}"
    )

    logger.info(energy_herm_str)
    logger.info(energy_raw_str)

    # 明确报警，别让模型安静地死
    if float(diag["log_real_span"]) < 1e-6:
        logger.warning(">>> WARNING: logΨ.real span ≈ 0，疑似 amplitude collapse / gauge plateau")

    if float(diag["trace_real_std"]) < 1e-8:
        logger.warning(">>> WARNING: trace(E_L).real std 很小，covariance 梯度能量项可能无信号")

    if float(diag["dlog_std_ratio"]) < 1e-6:
        logger.warning(">>> WARNING: dlogΨ batch variation 很小，参数响应接近常数方向")

    if float(diag["herm_error"]) > 1.0:
        logger.warning(">>> WARNING: E_L_mean 非 Hermitian 程度很大，能量诊断不可信")

    logger.info("#" + "-" * 79)


# ============================================================
# 9. History
# ============================================================

history = {
    "step": [],
    "loss": [],
    "energies_herm": [],
    "energies_raw": [],
    "E_Lmatrix": [],
    "samples": [],

    "grad_norm_raw": [],
    "grad_norm_update": [],
    "grad_norm_clipped": [],

    "log_real_std": [],
    "log_real_span": [],
    "trace_real_std": [],
    "trace_real_span": [],

    "dlog_std_norm": [],
    "dlog_mean_norm": [],
    "dlog_std_ratio": [],

    "psi_cond_mean": [],
    "psi_cond_max": [],

    "herm_error": [],
    "imag_norm": [],

    "unique_ratio": [],
    "valid_ratio": [],
}


# ============================================================
# 10. Training loop
# ============================================================

logger.info("\n" + "=" * 80)
logger.info("Start Training")
logger.info("=" * 80)

start_time = time.time()
n_skipped = 0

for step in range(N_ITER):

    # --------------------------------------------------------
    # Sampling
    # --------------------------------------------------------
    samples_raw, sampler_state = nes_sampler.sample(
        machine=total_machine,
        parameters=total_params,
        state=sampler_state,
        chain_length=N_SAMPLES_PER_CHAIN,
    )

    samples = samples_raw.reshape(-1, hi_ext.size)
    x_batch = samples.reshape(-1, K, SINGLE_SIZE)

    assert x_batch.shape[1:] == (K, SINGLE_SIZE), (
        f"x_batch.shape={x_batch.shape}, expected (-1, {K}, {SINGLE_SIZE})"
    )

    # --------------------------------------------------------
    # Gradient
    # --------------------------------------------------------
    grad_raw, loss_mean, E_L_mean = nes_vmc_gradient_stable(
        ha=ha,
        total_matrix_machine=total_matrix_machine,
        total_max_machine=total_max_machine,
        total_machine=total_machine,
        single_machine_list=single_machine_list,
        total_params=total_params,
        x_batch=x_batch,
    )

    assert E_L_mean.shape == (K, K), (
        f"E_L_mean.shape={E_L_mean.shape}, expected {(K, K)}"
    )

    grad_raw_flat, unravel_fn = ravel_pytree(grad_raw)
    grad_norm_raw = jnp.linalg.norm(grad_raw_flat)

    grad_finite = tree_all_finite(grad_raw)
    loss_finite = bool(jnp.isfinite(loss_mean))
    grad_explode = bool(grad_norm_raw > grad_skip_threshold)

    # --------------------------------------------------------
    # Natural gradient, optional
    # --------------------------------------------------------
    if Natural_Grad:
        qgt_reg_mat, _ = compute_qgt(
            total_machine,
            total_params,
            x_batch,
            diag_shift=qgt_diag_shift,
        )
        ng_flat = jnp.linalg.solve(qgt_reg_mat, grad_raw_flat)
        grad_update = unravel_fn(ng_flat)
        grad_norm_update = jnp.linalg.norm(ng_flat)
    else:
        grad_update = grad_raw
        grad_norm_update = grad_norm_raw

    # clip 后理论范数
    grad_norm_clipped = jnp.minimum(grad_norm_update, clip_norm)

    # --------------------------------------------------------
    # Diagnostics before possible update
    # --------------------------------------------------------
    need_print = (
        step % PRINT_EVERY == 0
        or step == N_ITER - 1
        or grad_explode
        or (not grad_finite)
        or (not loss_finite)
    )

    if need_print:
        diag = compute_diagnostics(
            total_params=total_params,
            x_batch=x_batch,
            samples=samples,
            E_L_mean=E_L_mean,
        )

        log_diagnostics(
            step=step,
            loss_mean=loss_mean,
            grad_norm_raw=grad_norm_raw,
            grad_norm_update=grad_norm_update,
            grad_norm_clipped=grad_norm_clipped,
            diag=diag,
        )

    # --------------------------------------------------------
    # Skip bad update
    # --------------------------------------------------------
    if (not grad_finite) or (not loss_finite) or grad_explode:
        n_skipped += 1
        logger.warning(
            f"【Step {step} Skip】bad update skipped | "
            f"grad_finite={grad_finite} | "
            f"loss_finite={loss_finite} | "
            f"grad_norm_raw={float(grad_norm_raw):.6e} | "
            f"skip_count={n_skipped}"
        )
        logger.info("#" + "=" * 79)
        continue

    # --------------------------------------------------------
    # Optimizer update
    # --------------------------------------------------------
    updates, opt_state = optimizer.update(
        grad_update,
        opt_state,
        total_params,
    )

    total_params = optax.apply_updates(total_params, updates)

    # --------------------------------------------------------
    # Save history
    # --------------------------------------------------------
    if need_print:
        history["step"].append(step)
        history["loss"].append(loss_mean)
        history["energies_herm"].append(diag["eig_vals_herm"])
        history["energies_raw"].append(diag["eig_vals_raw"])
        history["E_Lmatrix"].append(E_L_mean)
        history["samples"].append(samples)

        history["grad_norm_raw"].append(grad_norm_raw)
        history["grad_norm_update"].append(grad_norm_update)
        history["grad_norm_clipped"].append(grad_norm_clipped)

        history["log_real_std"].append(diag["log_real_std"])
        history["log_real_span"].append(diag["log_real_span"])

        history["trace_real_std"].append(diag["trace_real_std"])
        history["trace_real_span"].append(diag["trace_real_span"])

        history["dlog_std_norm"].append(diag["dlog_std_norm"])
        history["dlog_mean_norm"].append(diag["dlog_mean_norm"])
        history["dlog_std_ratio"].append(diag["dlog_std_ratio"])

        history["psi_cond_mean"].append(diag["psi_cond_mean"])
        history["psi_cond_max"].append(diag["psi_cond_max"])

        history["herm_error"].append(diag["herm_error"])
        history["imag_norm"].append(diag["imag_norm"])

        history["unique_ratio"].append(diag["unique_ratio"])
        history["valid_ratio"].append(diag["valid_ratio"])


end_time = time.time()

logger.info("\n" + "=" * 80)
logger.info("Training finished")
logger.info("=" * 80)
logger.info(f"Elapsed time = {end_time - start_time:.2f} sec")
logger.info(f"Skipped updates = {n_skipped}")
logger.info(f"Log file = {log_filename}")

print("\n" + "=" * 80)
print("训练完成")
print("=" * 80)
print(f"训练耗时：{end_time - start_time:.2f} 秒")
print(f"跳过更新次数：{n_skipped}")
print(f"日志文件：{log_filename}")


NES-VMC Stable Training | Diagnostic Version
K = 2
SINGLE_SIZE = 8
target_loss = -2.00505776
FCI E0 = -1.02613572 | E1 = -0.97892204
Hyperparameters
N_CHAINS = 32
N_WARMUP = 50
N_SAMPLES_PER_CHAIN = 200
SWEEP_SIZE = 30
N_ITER = 300
Natural_Grad = False
lr = 0.01
clip_norm = 1.0
grad_skip_threshold = 10.0
qgt_diag_shift = 0.1
DIAG_BATCH_SIZE = 128


H2 / 6-31G 基准
HF energy = -0.94605220 Ha
E0 = -1.02613572 Ha | excitation = 0.0000 eV
E1 = -0.97892204 Ha | excitation = 1.2848 eV
E2 = -0.66776157 Ha | excitation = 9.7519 eV
E3 = -0.60817046 Ha | excitation = 11.3734 eV
Hilbert 信息
K = 2
hi.size = 8
hi_ext.size = 16
SINGLE_SIZE = 8
target_loss = sum(E_fcis[:K]) = -2.00505776
Sampler edges
single_edges = [(3, 0), (3, 1), (3, 2), (7, 4), (7, 5), (7, 6)]
n_singles = 6
n_doubles = 9


/opt/miniconda3/envs/Netket/lib/python3.11/site-packages/jax/_src/ops/scatter.py:104: FutureWarning: scatter inputs have incompatible types: cannot safely cast value from dtype=complex128 to dtype=complex64 with jax_numpy_dtype_promotion=standard. In future JAX releases this will result in an error.
  warnings.warn(
Warmup
Warmup done.

Start Training
[Step    0]
Loss=0.57973486 | target=-2.00505776 | gap=+2.58479261
Grad | raw=1.9873e+00 | update=1.9873e+00 | clipped=1.0000e+00 | clip_norm=1.00e+00
logΨ.real | mean=-0.017794 | std=6.1989e-01 | span=4.0741e+00
logΨ.imag | mean=-0.031356 | std=1.9767e+00 | span=6.2507e+00
shift | mean=0.474339 | std=2.6605e-01 | span=8.2478e-01
cond(Ψ_stable) | first=2.6837e+00 | mean=4.5846e+00 | max=1.1980e+01
trace(E_L).real | mean=0.41584390 | std=8.8120e-01 | span=4.1630e+00
trace(E_L).imag | mean=-0.00867961 | std=2.4362e-01 | span=1.5193e+00
E_L_mean diagnostics | herm_error=4.5310e-01 | imag_norm=3.1562e-01 | valid_ratio=1.0000
sampler / dlogΨ |


训练完成
训练耗时：283.21 秒
跳过更新次数：35
日志文件：nes_vmc_H2_631G_K2_diagnostic.log


In [ ]:
import os, pickle

# 不存在则自动创建data文件夹
os.makedirs("./data", exist_ok=True)

with open('./data/history_0625.pkl', 'wb') as f:
    pickle.dump(history, f)


In [ ]:
history['samples'][-1]

In [ ]:
sampler_info(history['samples'][-1],K=3)

In [ ]:
E_L_mean = history['E_Lmatrix'][-1]
anti_herm = E_L_mean - E_L_mean.conj().T
anti_herm_norm = jnp.linalg.norm(anti_herm)
E_norm = jnp.linalg.norm(E_L_mean)

herm_error = anti_herm_norm / (E_norm + 1e-12)
herm_error

In [ ]:
eig_vals, eig_vecs = jnp.linalg.eig(E_L_mean)
eig_vals